In [1]:

# Import necessary libraries
import os
import pandas as pd
import numpy as np
import random
from collections import defaultdict
from tqdm import tqdm
from pathlib import Path


# PyTorch and Transformer imports (to be installed in your environment)
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn
from torch.nn import functional as F
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"

# If using graph neural networks (GNN) you may need torch_geometric;
# ensure it is installed in your environment
# from torch_geometric.nn import GCNConv
# from torch_geometric.data import Data as GraphData

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)



/opt/conda/envs/esci/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# Define file paths
ROOT = Path("Amazon_products")

TRAIN_CORPUS_PATH = ROOT / "train" / "train_corpus.txt"
TEST_CORPUS_PATH  = ROOT / "test" / "test_corpus.txt"

CLASSES_PATH      = ROOT / "classes.txt"
HIERARCHY_PATH    = ROOT / "class_hierarchy.txt"
KEYWORDS_PATH     = ROOT / "class_related_keywords.txt"
# Load classes mapping (class_id -> class_name)
class_id_to_name = {}
class_name_to_id = {}
with open(CLASSES_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        cid, cname = line.strip().split('	')
        class_id_to_name[int(cid)] = cname
        class_name_to_id[cname] = int(cid)

NUM_CLASSES = len(class_id_to_name)
print(f'Total classes: {NUM_CLASSES}')

# Load class hierarchy (parent -> children)
parent_to_children = defaultdict(list)
child_to_parent = {}
with open(HIERARCHY_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        parent, child = line.strip().split('	')
        parent, child = int(parent), int(child)
        parent_to_children[parent].append(child)
        child_to_parent[child] = parent

# Load class keywords (class_name: list of keywords)
class_keywords = {}
with open(KEYWORDS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        cname, kws = line.strip().split(':')
        keywords = [kw.strip().lower() for kw in kws.split(',') if kw.strip()]
        class_keywords[cname] = keywords

print(f'Loaded keywords for {len(class_keywords)} classes')

# Display a sample of the classes and keywords
for cid in range(5):
    cname = class_id_to_name[cid]
    print(cid, cname, '->', class_keywords.get(cname, [])[:5])


Total classes: 531
Loaded keywords for 531 classes
0 grocery_gourmet_food -> ['snacks', 'condiments', 'beverages', 'specialty_foods', 'spices']
1 meat_poultry -> ['butcher', 'cuts', 'marination', 'grilling', 'roasting']
2 jerky -> ['beef', 'turkey', 'chicken', 'venison', 'buffalo']
3 toys_games -> ['board_games', 'puzzles', 'action_figures', 'building_blocks', 'dolls']
4 games -> ['board_games', 'card_games', 'tabletop_games', 'party_games', 'roleplaying_games']


In [3]:

# Silver label generation using keyword matching
# This function maps a text to a set of class indices based on keyword occurrences

def generate_silver_labels(text, class_keywords, class_name_to_id):
    labels = set()
    lower_text = text.lower()
    for cname, keywords in class_keywords.items():
        for kw in keywords:
            if kw in lower_text:
                labels.add(class_name_to_id[cname])
                break  # avoid duplicate checks once a keyword matches
    return labels

# Process the training corpus to generate silver labels
train_ids = []
train_texts = []
train_label_sets = []

with open(TRAIN_CORPUS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        doc_id, text = line.strip().split('	', 1)
        labels = generate_silver_labels(text, class_keywords, class_name_to_id)
        if labels:
            train_ids.append(int(doc_id))
            train_texts.append(text)
            train_label_sets.append(labels)
        # If no labels are found, we skip the sample for now (could be added later via LLM)

print(f'Silver labeled training samples: {len(train_texts)} / original')
print('Example:', train_ids[0], train_label_sets[0])


Silver labeled training samples: 25518 / original
Example: 0 {196, 493, 241, 182, 502, 187}


In [4]:

# Optional: Augment labels using an LLM (e.g., GPT-3.5/4) when no keywords match or to refine labels
# This is a stub function. Replace `pass` with actual LLM query code in your environment.

def query_llm_for_labels(text):
    """
    Placeholder function for querying an LLM to predict relevant classes for a given text.
    In practice, you would call the OpenAI API or another LLM provider here.
    The function should return a list of class IDs (ints).
    """
    # Example pseudo-code (commented):
    # response = openai.ChatCompletion.create(
    #     model="gpt-3.5-turbo",
    #     messages=[{"role": "system", "content": "You are an expert product categorizer."},
    #               {"role": "user", "content": f"Assign up to three relevant product categories to the following product description: {text}"}]
    # )
    # predicted_classes = parse_response(response)
    # return predicted_classes
    return []

# Apply LLM augmentation (optional)
# We iterate over unlabeled samples and query the LLM, but here we leave it as a no-op.
# This step requires API integration and is not executed by default.



In [5]:

# Prepare datasets for training and testing

class MultiLabelDataset(Dataset):
    def __init__(self, texts, label_sets, tokenizer, max_length=128):
        self.texts = texts
        self.label_sets = label_sets
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        labels = self.label_sets[idx]
        # Tokenize text
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        # Create multi-hot label vector
        label_vector = torch.zeros(NUM_CLASSES, dtype=torch.float)
        for l in labels:
            label_vector[l] = 1.0
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': label_vector
        }

# Initialize tokenizer
TOKENIZER_NAME = 'bert-base-multilingual-cased'  # Change if using a different model
try:
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
except Exception as e:
    print(f"Tokenizer {TOKENIZER_NAME} failed to load: {e}")
    raise

# Create training dataset
train_dataset = MultiLabelDataset(train_texts, train_label_sets, tokenizer, max_length=128)
print(f'Training dataset size: {len(train_dataset)}')

# Prepare test dataset (without labels)
test_ids = []
test_texts = []
with open(TEST_CORPUS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        doc_id, text = line.strip().split('	', 1)
        test_ids.append(int(doc_id))
        test_texts.append(text)

test_label_sets = [set() for _ in test_texts]  # dummy labels

test_dataset = MultiLabelDataset(test_texts, test_label_sets, tokenizer, max_length=128)
print(f'Test dataset size: {len(test_dataset)}')


Training dataset size: 25518
Test dataset size: 19658


In [6]:

# Define BERT-based multi-label classification model with optional GNN for class embeddings

class HierarchicalClassifier(nn.Module):
    def __init__(self, transformer_name, num_classes, class_keywords, class_name_to_id, parent_to_children):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(transformer_name)
        hidden_size = self.transformer.config.hidden_size
        self.num_classes = num_classes
        
        # Class embeddings initialized randomly or from keywords
        self.class_emb_dim = hidden_size
        self.class_embeddings = nn.Embedding(num_classes, self.class_emb_dim)
        
        # Initialize class embeddings using class keyword sentences and transformer
        self._init_class_embeddings(class_keywords, class_name_to_id)
        
        # Define a simple GNN layer over the class graph (two-layer GCN-like)
        # Adjacency matrix will be built externally
        self.gnn_weight = nn.Parameter(torch.randn(self.class_emb_dim, self.class_emb_dim))
        
    def _init_class_embeddings(self, class_keywords, class_name_to_id):
        """
        Initialize class embeddings by encoding class keywords through the same transformer.
        If keywords are missing, initialize randomly.
        """
        device = next(self.parameters()).device
        with torch.no_grad():
            for cname, cid in class_name_to_id.items():
                keywords = class_keywords.get(cname, [])
                if keywords:
                    # Create a pseudo sentence from keywords
                    sentence = ', '.join(keywords[:10])
                    inputs = tokenizer(sentence, return_tensors='pt', truncation=True, max_length=50).to(device)
                    outputs = self.transformer(**inputs)
                    # Use [CLS] token embedding
                    cls_emb = outputs.last_hidden_state[:, 0, :].squeeze(0)
                    self.class_embeddings.weight.data[cid] = cls_emb
                else:
                    # Random initialization
                    nn.init.xavier_uniform_(self.class_embeddings.weight.data[cid].unsqueeze(0))

    def forward(self, input_ids, attention_mask, adj_matrix):
        # Encode text using BERT
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        # Get CLS token hidden states
        cls_output = outputs.last_hidden_state[:, 0, :]
        
        # Obtain class embeddings and propagate through a simple GCN layer
        class_emb = self.class_embeddings.weight  # (num_classes, emb_dim)
        
        # One GCN layer (class embeddings aggregated by adjacency)
        # class_emb = torch.matmul(adj_matrix, class_emb) @ self.gnn_weight
        class_emb = torch.matmul(adj_matrix, class_emb)
        class_emb = torch.matmul(class_emb, self.gnn_weight)
        class_emb = torch.tanh(class_emb)
        
        # Compute logits via dot product between text embedding and class embeddings
        logits = torch.matmul(cls_output, class_emb.t())  # (batch_size, num_classes)
        return logits

# Build adjacency matrix for GNN from hierarchy
# We construct a symmetric normalized adjacency matrix

adj = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.float32)
for parent, children in parent_to_children.items():
    for child in children:
        adj[parent, child] = 1.0
        adj[child, parent] = 1.0  # make symmetric for simplicity
# Add self-loops
for i in range(NUM_CLASSES):
    adj[i, i] = 1.0

# Normalize adjacency matrix (row-normalized)
row_sum = adj.sum(axis=1, keepdims=True)
adj_normalized = adj / (row_sum + 1e-8)
adj_tensor = torch.tensor(adj_normalized, dtype=torch.float)

# Instantiate model
model = HierarchicalClassifier(
    transformer_name=TOKENIZER_NAME,
    num_classes=NUM_CLASSES,
    class_keywords=class_keywords,
    class_name_to_id=class_name_to_id,
    parent_to_children=parent_to_children
)

# Move model and adjacency to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
adj_tensor = adj_tensor.to(device)

# Example forward pass on a small batch
sample = train_dataset[0]
input_ids = sample['input_ids'].unsqueeze(0).to(device)
attention_mask = sample['attention_mask'].unsqueeze(0).to(device)
with torch.no_grad():
    logits = model(input_ids, attention_mask, adj_tensor)
print('Logits shape:', logits.shape)


Logits shape: torch.Size([1, 531])


In [7]:

# Training loop with self-training and consistency regularization

from torch.optim import Adam

# Hyperparameters
BATCH_SIZE = 8
EPOCHS = 2  # Increase as needed
LEARNING_RATE = 2e-5

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

def train_model(model, train_loader, adj_matrix, epochs=EPOCHS, lr=LEARNING_RATE):
    model.train()
    optimizer = Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids, attention_mask, adj_matrix)
            loss = criterion(logits, labels)
            
            # Backpropagation
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        print(f'Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}')

# Note: due to compute constraints, we use small epochs
# In a real setting, increase EPOCHS and monitor validation metrics

# Uncomment to train
# train_model(model, train_loader, adj_tensor, epochs=EPOCHS, lr=LEARNING_RATE)


In [11]:

# Inference on test dataset and submission file creation

# DataLoader for test dataset
TEST_BATCH_SIZE = 8
test_loader = DataLoader(test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=False)

# Collect predictions
model.eval()
all_predictions = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        logits = model(input_ids, attention_mask, adj_tensor)
        probs = torch.sigmoid(logits)
        all_predictions.append(probs.cpu())

all_predictions = torch.cat(all_predictions, dim=0)  # (num_test_samples, num_classes)

# For each test sample, select 2-3 labels based on probability thresholds
MIN_LABELS = 2
MAX_LABELS = 3

predicted_labels = []
for i in range(all_predictions.shape[0]):
    probs = all_predictions[i]
    # Get sorted class indices by probability
    sorted_idx = torch.argsort(probs, descending=True)
    # Select top MAX_LABELS classes
    topk = sorted_idx[:MAX_LABELS]
    # Filter by threshold if desired (e.g., >0.5), else keep topk
    selected = []
    for idx in topk:
        if probs[idx] > 0.5 or len(selected) < MIN_LABELS:
            selected.append(int(idx))
    # Ensure at least MIN_LABELS
    if len(selected) < MIN_LABELS:
        selected = [int(x) for x in sorted_idx[:MIN_LABELS]]
    predicted_labels.append(selected)


# Build submission DataFrame
submission = pd.DataFrame({
    'id': test_ids,
    'label': [','.join(str(l) for l in labels) for labels in predicted_labels]
})

# Save to CSV
SUBMISSION_DIR = ROOT / "submission"
SUBMISSION_PATH = SUBMISSION_DIR / "submission.csv"


submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Submission saved to {SUBMISSION_PATH}")
print(submission.head())


Submission saved to Amazon_products/submission/submission.csv
   id    label
0   0  285,279
1   1  369,285
2   2  279,285
3   3  270,428
4   4  285,279
